In [ ]:
import pydicom, os

# dossier sur mon bureau a changer 
#dossier = "../DICOM_HCL/25EH09982-1-0-L-4_5_121428/"
dossier = "../DICOM_HCL/25EH10679-1-CONG01-L-16_5_121935"
fichiers = sorted([f for f in os.listdir(dossier) if f.endswith('.dcm')])
gros = max(fichiers, key=lambda f: os.path.getsize(os.path.join(dossier, f)))

types_compression = {
    '1.2.840.10008.1.2.4.50': 'JPEG Baseline (lossy)',
    '1.2.840.10008.1.2.4.51': 'JPEG Extended (lossy)',
    '1.2.840.10008.1.2.4.57': 'JPEG Lossless',
    '1.2.840.10008.1.2.4.90': 'JPEG 2000 Lossless',
    '1.2.840.10008.1.2.4.91': 'JPEG 2000',
}

non_compresses = ['1.2.840.10008.1.2', '1.2.840.10008.1.2.1', '1.2.840.10008.1.2.2']

In [31]:
for f in fichiers:
    path = os.path.join(dossier, f)
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    ts = str(ds.file_meta.TransferSyntaxUID)
    comp = 'compressé' if ts not in non_compresses else 'non compressé'
    nf = getattr(ds, 'NumberOfFrames', 1) or 1
    
    print(f"{ds.Columns:>5}x{ds.Rows:<5} | {os.path.getsize(path)/1024/1024:6.1f} Mo | {nf:>7} frames | {types_compression.get(ts, ts)} ({comp})")

  256x256   |   12.7 Mo |      35 frames | JPEG Baseline (lossy) (compressé)
  256x256   |   15.5 Mo |     476 frames | JPEG Baseline (lossy) (compressé)
  256x256   |   59.9 Mo |    7504 frames | JPEG Baseline (lossy) (compressé)
  602x608   |   13.6 Mo |       1 frames | 1.2.840.10008.1.2.1 (non compressé)
 1481x599   |   12.6 Mo |       1 frames | JPEG Baseline (lossy) (compressé)
  256x256   |  846.2 Mo |  119168 frames | JPEG Baseline (lossy) (compressé)
 1920x1141  |   12.7 Mo |       1 frames | JPEG Baseline (lossy) (compressé)


In [32]:
ds = pydicom.dcmread(os.path.join(dossier, gros))

rows, cols = ds.Rows, ds.Columns
spp, bits = ds.SamplesPerPixel, ds.BitsAllocated
nf = ds.NumberOfFrames
taille_fichier = os.path.getsize(os.path.join(dossier, gros))

# taille non compressée = rows * cols * canaux * (bits/8)
taille_tuile_raw = rows * cols * spp * (bits // 8)
taille_totale_raw = taille_tuile_raw * nf
ratio_reel = taille_totale_raw / taille_fichier

print("Fichier principal (lame entière) :")
print(f"  {ds.TotalPixelMatrixColumns} x {ds.TotalPixelMatrixRows} px")
print(f"  {nf:,} tuiles de {cols}x{cols} px, {ds.PhotometricInterpretation}, {bits}-bit")
print(f"  Fichier = {taille_fichier/1024/1024:.0f} Mo")
print()
print(f"  Compression : JPEG Baseline (ISO 10918-1) — avec perte")
print(f"  Ratio annoncé (tag DICOM) : {ds.LossyImageCompressionRatio}:1")
print(f"  Ratio mesuré              : {ratio_reel:.1f}:1")
print(f"  Taille moyenne par tuile  : {taille_fichier/nf:,.0f} o (non compressé = {taille_tuile_raw:,} o)")
print()
print(f"  Niveau de compression : TRÈS ÉLEVÉ")
print(f"  Seulement {(1/ratio_reel)*100:.1f}% des données conservées")

Fichier principal (lame entière) :
  114484 x 68070 px
  119,168 tuiles de 256x256 px, YBR_FULL_422, 8-bit
  Fichier = 846 Mo

  Compression : JPEG Baseline (ISO 10918-1) — avec perte
  Ratio annoncé (tag DICOM) : 10:1
  Ratio mesuré              : 26.4:1
  Taille moyenne par tuile  : 7,446 o (non compressé = 196,608 o)

  Niveau de compression : TRÈS ÉLEVÉ
  Seulement 3.8% des données conservées


In [34]:
print("Scanner :", ds.Manufacturer, ds.ManufacturerModelName)
print("Logiciel :", ds.SoftwareVersions)
print("Acquisition :", ds.AcquisitionDateTime)
print("Image Type :", ' > '.join(ds.ImageType))

Scanner : Leica Biosystems GT450
Logiciel : 1.4.1
Acquisition : 20250723121936+0200
Image Type : ORIGINAL > PRIMARY > VOLUME > NONE


In [35]:
import pydicom, os
f = "../DICOM_HCL/25EH09982-1-0-L-4_5_121428/"
big = max([x for x in os.listdir(f) if x.endswith('.dcm')], key=lambda x: os.path.getsize(f + x))
ds = pydicom.dcmread(f + big)
# Métadonnées
print(ds.file_meta.TransferSyntaxUID)
print(ds.file_meta.TransferSyntaxUID.name)
print(ds.PhotometricInterpretation)
print(ds.Rows, "x", ds.Columns, "x", ds.NumberOfFrames)
print("LossyImageCompression:", ds.LossyImageCompression)
print("LossyImageCompressionRatio:", ds.LossyImageCompressionRatio)
# Ratio calculé
raw_tuile = ds.Rows * ds.Columns * ds.SamplesPerPixel * (ds.BitsAllocated // 8)
raw_total = raw_tuile * ds.NumberOfFrames
taille_fichier = os.path.getsize(f + big)
print(f"Ratio réel: {raw_total / taille_fichier:.1f}:1")

1.2.840.10008.1.2.4.50
JPEG Baseline (Process 1)
YBR_FULL_422
256 x 256 x 120506
LossyImageCompression: 01
LossyImageCompressionRatio: 10
Ratio réel: 33.6:1
